# Data Modeling - Interruptions

### Interruption Table
The dataset is derived from post text data. The initial structure of the table is created using the cleaned text column from the `interruption_silver` table.

From this base, additional columns will be be added as more fields are extracted in the following transformations.

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_gold AS
SELECT
  text
FROM beneco_pipeline.silver.interruption_silver;
SELECT * FROM beneco_pipeline.gold.interruption_gold;

### Type Capturing
The type of interruption will fall under scheduled, unscheduled and emergency. 

This information can be extracted using regular expressions, as the interruption type is consistently stated in each advisory post.

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_gold
SELECT
    text,
    regexp_extract(text,r'(\w+)\s*Power Interruption', 1) AS type
FROM beneco_pipeline.gold.interruption_gold;

-- Query to check if all records have a type
SELECT * FROM beneco_pipeline.gold.interruption_gold WHERE type IS NULL;

### Start Datetime Capturing
Since the posts do not follow a strict format, the start date time capturing will need to accout for the different ways the date and start time of the interruption is stated. The datetime format will be in `yyyy-MM-dd HH:mm`

**Case 1:** 
```
DATE: Sunday, February 22, 2026

TIME STARTED: 09:00 AM 
or STARTED: Sunday, February 22, 2026 at 09:00 AM
or TIME: 9:00 AM to 5:00 PM
```

**Case 2:**
```
DATE AND TIME STARTED: Sunday, February 22, 2026 at 09:00 AM 
```

The date is captured first then the time is concatinated. For the example above, `February 22, 2026 09:00 AM` then convert to `2026-02-22 09:00`

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_gold
AS SELECT
    text,
    type,
    date_format(try_to_timestamp(
    CASE 
        -- for DATE: Sunday, February 22, 2026
        WHEN text RLIKE r'(?i)DATE:\s*\w+\s*,?\s*(\w+\s*\d{1,2},\s*\d{4})' THEN (
            -- Date capture
            regexp_extract(text,r'(?i)DATE:(?:\s*\w+\s*,)?\s*(\w+\s*\d{1,2},\s*\d{4})',1) ||' '|| 
            -- time capture for START(ED): XX:XX AM/PM
            regexp_extract(text,r'(?i)START(?:ED)?\s*:(?:\s*)?(\d{1,2}:\s*\d{2}\s*[AP]M)',1) || 
            -- for case: START(ED): (day), MMMM dd, yyyy at XX:XX AM/PM
            regexp_extract(text,r'(?i)START(?:ED)?\s*:(?:\s*)?\w+\s*,?\s*\w+\s*\d{1,2},\s*\d{4}\s*;?(?:AT)?\s(\d{1,2}:\d{2}\s*[AP]M)',1) || 
            -- for case TIME: HH:MM AM/PM
            regexp_extract(text,r'(?i)TIME\s*:\s*(\d{1,2}:\d{2}\s*[AP]M)',1))

        -- for DATE AND TIME STARTED: Tuesday, January 06, 2026 at 06:45 PM 
        WHEN TEXT RLIKE r'(?i)\d{1,2}:\d{2}\s(AM|PM)' THEN (
            -- Date capture
            regexp_extract(text,r'(?i)START(?:ED)?\s*:(?:\s*\w+\s*,)?\s*(\w+\s*\d{1,2},\s*\d{4}),?\s*;?(?:AT)?\s\d{1,2}:\d{2}\s*[AP]M',1)||' '|| 
            -- time capture
            regexp_extract(text,r'(?i)START(?:ED)?\s*:(?:\s*\w+\s*,)?\s*\w+\s*\d{1,2},\s*\d{4},?\s*;?(?:AT)?\s(\d{1,2}:\d{2}\s*[AP]M)',1))
        ELSE 'no format'
    END
    ,'MMMM d, yyyy h:mm a'), 'yyyy-MM-dd HH:mm')
    AS start
FROM beneco_pipeline.gold.interruption_gold;

-- Query to check if any date is not captured properly
SELECT * FROM beneco_pipeline.gold.interruption_gold WHERE start IS NULL;

### End Datetime Capturing
The end date time will also have different capture cases, however if there was no update on the restore time, then the end record will be null. 

**Case 1:**
```
DATE AND TIME RESTORED/ENERGIZED: Sunday, February 22, 2026 at 05:00 PM 
```

**Case 2:** 
```
DATE: Sunday, February 22, 2026
RESTORED/ENERGIZED: 5:00 PM
or TIME: 9:00 AM to 5:00 PM
or ENERGIZED AS OF: 5:00 PM
```




In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_gold
AS SELECT
    text, 
    type,
    start,
    date_format(try_to_timestamp(
    CASE 
        --for DATE AND TIME RESTORED: (Tuesday,)? January 06, 2026 at 06:45 PM
        WHEN text RLIKE r'(?i)(?:RESTOR(?:ED|ATION|E)?|ENERGIZED|ENDED)\s*:(?:\s*\w+\s*,)?\s*(\w+\s*\d{1,2},\s*\d{4}),?\s*;?(?:AT|@)?\s\d{1,2}:\d{2}\s*[AP]M' THEN (
            -- Date Capture
            regexp_extract(text,r'(?i)(?:RESTOR(?:ED|ATION|E)?|ENERGIZED|ENDED)\s*:(?:\s*\w+\s*,)?\s*(\w+\s*\d{1,2},\s*\d{4}),?\s*;?(?:AT|@)?\s\d{1,2}:\d{2}\s*[AP]M',1)||' '||
            -- Time Capture
            regexp_extract(text,r'(?i)(?:RESTOR(?:ED|ATION|E)?|ENERGIZED|ENDED)\s*:(?:\s*\w+\s*,)?\s*\w+\s*\d{1,2},\s*\d{4},?\s*;?(?:AT|@)?\s(\d{1,2}:\d{2}\s*[AP]M)',1))
        
        -- for DATE: Sunday, February 22, 2026
        WHEN text RLIKE r'(?i)DATE:(?:\s*\w+\s*,)?\s*(\w+\s*\d{1,2},\s*\d{4})' THEN (
            -- Date capture
            regexp_extract(text,r'(?i)DATE:(?:\s*\w+\s*,)?\s*(\w+\s*\d{1,2},\s*\d{4})',1) ||' '|| 
            CASE
                -- for case: RESTORED: XX:XX AM/PM
                WHEN text RLIKE r'(?i)(?:RESTOR(?:ED|ATION|E)?|ENERGIZED|ENDED)\s*:(?:\s*)?(\d{1,2}:\s*\d{2}\s*[AP]M)' THEN (
                regexp_extract(text,r'(?i)(?:RESTOR(?:ED|ATION|E)?|ENERGIZED|ENDED)\s*:(?:\s*)?(\d{1,2}:\s*\d{2}\s*[AP]M)',1))
                -- for case: TIME: HH:MM AM/PM to HH:MM AM/PM OR AS OF HH:MM AM/PM
                WHEN text RLIKE r'(?i)(?:TO|OF)\s*(\d{1,2}:\d{2}\s*[AP]M)' THEN (
                regexp_extract(text,r'(?i)(?:TO|OF)\s*(\d{1,2}:\d{2}\s*[AP]M)',1))
                ELSE 'no end'
            END
        )
        ELSE 'no capture'
    END
    , 'MMMM d, yyyy h:mm a'), 'yyyy-MM-dd HH:mm') 
    AS end
FROM beneco_pipeline.gold.interruption_gold;

-- Query to check any end dates are not captured or post does not have restoration update
SELECT text, end FROM beneco_pipeline.gold.interruption_gold WHERE end IS NULL;
-- AND TEXT RLIKE '((?i)(?:RESTOR(?:ED|ATION|E)?|ENERGIZED|ENDED)):';

### Remark Capturing
Power outage remarks may be the purpose of the outage in the case of scheduled outages or cause in the case of unscheduled/emergency outages. The status of the outage may also be included in the capture.

In [0]:
%sql
CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_gold
AS SELECT
    text, 
    type,
    start,
    end,
    CASE 
      -- Purpose + Status
      WHEN text RLIKE r'(?i)(PURPOSE):?' THEN(
        regexp_extract(text,r'(?i)(PURPOSE:?[^\n\r]+)',1)
        ||'\n'||regexp_extract(text,r'(?i)(STATUS:?[^\n\r]+)',1))
      -- Cause + Status
      WHEN text RLIKE r'(?i)CAUSE:?' THEN(
        regexp_extract(text,r'(?i)(CAUSE[^\n\r]+)',1)
        ||'\n'||regexp_extract(text,r'(?i)(STATUS:?[^\n\r]+)',1))
      -- Status
      WHEN text RLIKE r'(?i)STATUS:?' THEN(
        regexp_extract(text,r'(?i)(STATUS:?[^\n\r]+)',1))
      ELSE 'No remark'
    END
    AS remark
FROM beneco_pipeline.gold.interruption_gold;

-- Query check if remark is not captured or there is no remark in the post
SELECT text, remark FROM beneco_pipeline.gold.interruption_gold
WHERE remark LIKE '%No remark%';

### Primary key
Add id column as the primary key to reference in queries to do this, the following steps are made:

* Save 'interruption_gold' into temp table 'interruption_temp'.
* Recreate 'interruption_gold' and add the id column as primary key.
* Insert `interruption_temp` into the new  `interruption_gold` table.
* Drop the `interruption_temp` table.

In [0]:
%sql
-- Save data to a separate temp table
CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_temp AS
SELECT * FROM beneco_pipeline.gold.interruption_gold;

-- Recreate main table with primary key and change start and end to timestamp
CREATE OR REPLACE TABLE beneco_pipeline.gold.interruption_gold (
  id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  text STRING,
  type STRING,
  start STRING,
  end STRING,
  remark STRING
);

-- Insert from backup
INSERT INTO beneco_pipeline.gold.interruption_gold (text, type, start, end, remark)
SELECT text, type, start, end, remark FROM beneco_pipeline.gold.interruption_temp;

-- Clean up backup
DROP TABLE beneco_pipeline.gold.interruption_temp;

In [0]:
%sql
SELECT * FROM beneco_pipeline.gold.interruption_gold;